# RetainFlow - Dashboard De Drift

Notebook d'analyse du drift entre `train`, `validation`, `backtest` et `test`. Les cellules appellent uniquement les classes et fonctions de `src/retainflow`.

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from IPython.display import HTML, display

from retainflow.config import load_churn_model_config
from retainflow.evaluation.drift import DriftAnalyzer, DriftDashboardBuilder
from retainflow.logging import get_logger
from retainflow.pipelines.drift_dashboard import ChurnDriftDashboardPipeline


In [ ]:
logger = get_logger("retainflow.notebooks.drift")
logger.info("Drift notebook started")


## 2. Charger La Configuration

In [ ]:
config = load_churn_model_config("config/churn_model.yml")
config


## 3. Creer La Classe Pipeline

In [ ]:
pipeline = ChurnDriftDashboardPipeline(
    config=config,
    analyzer=DriftAnalyzer(),
    dashboard_builder=DriftDashboardBuilder(),
)
pipeline


## 4. Charger Les Donnees Depuis PostgreSQL

In [ ]:
raw_dataset = pipeline.load_raw_dataset()
raw_dataset.shape


In [ ]:
raw_dataset.groupby("split_name").size().reset_index(name="rows")


## 5. Construire Le Dataset De Features

In [ ]:
drift_dataset = pipeline.build_feature_dataset(raw_dataset)
drift_dataset.shape


In [ ]:
drift_dataset.groupby("split_name")["churn_label"].agg(
    rows="size",
    churn_rate="mean",
).reset_index()


## 6. Analyser Le Drift

In [ ]:
drift_report, drift_summary = pipeline.analyze(drift_dataset)
drift_summary


In [ ]:
drift_report.head(20)


## 7. Variables Les Plus Instables Vs Train

In [ ]:
reference_drift = drift_report[drift_report["is_reference_comparison"]].copy()
reference_drift.sort_values(["severity_rank", "psi"], ascending=[False, False]).head(30)


## 8. Sauvegarder Le Dashboard

In [ ]:
result = pipeline.save(drift_report, drift_summary)
result


## 9. Afficher Le Dashboard Dans Le Notebook

In [ ]:
dashboard_path = config.drift_dashboard_path
display(HTML(dashboard_path.read_text(encoding="utf-8")))
dashboard_path


## 10. Relire Le JSON Pour L'Agent

In [ ]:
with config.drift_summary_path.open(encoding="utf-8") as file:
    drift_payload_for_agent = json.load(file)

drift_payload_for_agent["top_drift_features"][:10]
